# Stopping Rules Comparison

This notebook provides an in-depth comparison of different stopping rules for systematic reviews.

## Stopping Rules Covered
1. **Bayesian Stopping**: Posterior probability of achieving target recall
2. **SPRT**: Sequential Probability Ratio Test
3. **SAFE**: Stop After First Estimate procedure
4. **Consecutive Irrelevant**: Simple threshold-based rule

In [ ]:
import asreview_5star as a5s
import numpy as np
import matplotlib.pyplot as plt

# For reproducibility
np.random.seed(42)

## Simulate a Screening Session

Let's simulate screening a dataset with known prevalence.

In [ ]:
# Simulation parameters
n_total = 5000
true_prevalence = 0.02  # 2% relevant

# Generate dataset (1 = relevant, 0 = irrelevant)
# In reality, active learning would prioritize relevant documents
# Here we simulate random screening
relevance = np.random.choice([0, 1], size=n_total, p=[1-true_prevalence, true_prevalence])

print(f"Total documents: {n_total}")
print(f"True relevant: {relevance.sum()}")
print(f"True prevalence: {relevance.mean():.2%}")

## Compare Stopping Rules During Screening

In [ ]:
# Track stopping rule results during screening
checkpoints = list(range(100, n_total + 1, 100))

results = {
    'screened': [],
    'relevant_found': [],
    'consecutive_irrelevant': [],
    'bayesian_conf': [],
    'bayesian_stop': [],
    'sprt_stop': [],
    'sprt_decision': [],
    'safe_conf': [],
    'safe_stop': [],
    'consec_stop': []
}

n_relevant = 0
consecutive_irrelevant = 0

for i, doc in enumerate(relevance):
    if doc == 1:
        n_relevant += 1
        consecutive_irrelevant = 0
    else:
        consecutive_irrelevant += 1
    
    n_screened = i + 1
    
    if n_screened in checkpoints:
        # Bayesian
        bayesian = a5s.bayesian_stopping(
            n_screened=n_screened,
            n_relevant=n_relevant,
            n_total=n_total,
            target_recall=0.95
        )
        
        # SPRT
        sprt = a5s.sprt_stopping(
            n_screened=n_screened,
            n_relevant=n_relevant
        )
        
        # SAFE
        safe = a5s.safe_stopping(
            n_screened=n_screened,
            n_relevant=n_relevant,
            consecutive_irrelevant=consecutive_irrelevant,
            n_total=n_total
        )
        
        # Consecutive
        consec = a5s.consecutive_irrelevant_stopping(
            consecutive_count=consecutive_irrelevant,
            threshold=50,
            n_relevant=n_relevant
        )
        
        results['screened'].append(n_screened)
        results['relevant_found'].append(n_relevant)
        results['consecutive_irrelevant'].append(consecutive_irrelevant)
        results['bayesian_conf'].append(bayesian.confidence)
        results['bayesian_stop'].append(bayesian.should_stop)
        results['sprt_stop'].append(sprt.should_stop)
        results['sprt_decision'].append(sprt.details.get('decision', 'continue'))
        results['safe_conf'].append(safe.confidence)
        results['safe_stop'].append(safe.should_stop)
        results['consec_stop'].append(consec.should_stop)

## Visualize Stopping Rule Performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Bayesian confidence
ax1 = axes[0, 0]
ax1.plot(results['screened'], results['bayesian_conf'], 'b-', linewidth=2)
ax1.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
ax1.set_xlabel('Documents Screened')
ax1.set_ylabel('Confidence')
ax1.set_title('Bayesian Stopping Confidence')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: SAFE confidence
ax2 = axes[0, 1]
ax2.plot(results['screened'], results['safe_conf'], 'g-', linewidth=2)
ax2.axhline(y=1.0, color='r', linestyle='--', label='Stop threshold')
ax2.set_xlabel('Documents Screened')
ax2.set_ylabel('Confidence')
ax2.set_title('SAFE Stopping Confidence')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Consecutive irrelevant count
ax3 = axes[1, 0]
ax3.plot(results['screened'], results['consecutive_irrelevant'], 'purple', linewidth=2)
ax3.axhline(y=50, color='r', linestyle='--', label='Threshold (50)')
ax3.set_xlabel('Documents Screened')
ax3.set_ylabel('Consecutive Irrelevant')
ax3.set_title('Consecutive Irrelevant Documents')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Cumulative relevant found
ax4 = axes[1, 1]
ax4.plot(results['screened'], results['relevant_found'], 'orange', linewidth=2)
ax4.axhline(y=relevance.sum(), color='r', linestyle='--', label=f'True total ({relevance.sum()})')
ax4.set_xlabel('Documents Screened')
ax4.set_ylabel('Relevant Found')
ax4.set_title('Cumulative Relevant Documents Found')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## When Did Each Rule Suggest Stopping?

In [ ]:
def find_first_stop(stops, screened):
    for i, stop in enumerate(stops):
        if stop:
            return screened[i]
    return None

bayesian_first = find_first_stop(results['bayesian_stop'], results['screened'])
sprt_first = find_first_stop(results['sprt_stop'], results['screened'])
safe_first = find_first_stop(results['safe_stop'], results['screened'])
consec_first = find_first_stop(results['consec_stop'], results['screened'])

print("First Stopping Point for Each Rule:")
print(f"  Bayesian:    {bayesian_first or 'Never'} ({bayesian_first/n_total*100 if bayesian_first else 'N/A'}% of dataset)")
print(f"  SPRT:        {sprt_first or 'Never'} ({sprt_first/n_total*100 if sprt_first else 'N/A'}% of dataset)")
print(f"  SAFE:        {safe_first or 'Never'} ({safe_first/n_total*100 if safe_first else 'N/A'}% of dataset)")
print(f"  Consecutive: {consec_first or 'Never'} ({consec_first/n_total*100 if consec_first else 'N/A'}% of dataset)")

## Recall at Stopping Points

In [ ]:
def get_recall_at_stop(stop_point, screened, relevant_found, true_total):
    if stop_point is None:
        return None
    idx = screened.index(stop_point)
    return relevant_found[idx] / true_total

true_total = relevance.sum()

print("Recall at Each Stopping Point:")
if bayesian_first:
    recall = get_recall_at_stop(bayesian_first, results['screened'], results['relevant_found'], true_total)
    print(f"  Bayesian:    {recall:.2%}")
if sprt_first:
    recall = get_recall_at_stop(sprt_first, results['screened'], results['relevant_found'], true_total)
    print(f"  SPRT:        {recall:.2%}")
if safe_first:
    recall = get_recall_at_stop(safe_first, results['screened'], results['relevant_found'], true_total)
    print(f"  SAFE:        {recall:.2%}")
if consec_first:
    recall = get_recall_at_stop(consec_first, results['screened'], results['relevant_found'], true_total)
    print(f"  Consecutive: {recall:.2%}")

## Recommendations

| Rule | Best For | Caution |
|------|----------|--------|
| **Bayesian** | High-stakes reviews requiring formal probability | Computationally intensive |
| **SPRT** | Quick screening with known prevalence expectations | Requires good prior estimates |
| **SAFE** | Balanced approach with adaptive thresholds | Needs at least some relevant found |
| **Consecutive** | Simple screening with clear stopping point | May stop too early with clustered relevant docs |

### General Guidance
- Use **multiple rules** and require consensus
- Never stop before finding at least **one relevant document**
- Consider the **cost of missing relevant documents** vs. **screening effort**